# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Setup

This notebook brings together the work from the previous weeks into one research paper.

Google Colab starts with a fresh runtime, so the repository and dataset are loaded again here instead of depending on variables from earlier notebooks.

The notebook covers:

1. Research question
2. Data
3. Methodology
4. Results compared with a baseline
5. Limitations
6. Ranked recommendations
7. Charts and tables for the final paper

The final paper will be deployed separately as a public research page.

The analysis is intended for decision support, not as a production system.

In [ ]:
# Clone the repository into the fresh Colab runtime
!git clone https://github.com/martindiarua/ML_01.git

# Move into the repository
%cd /content/ML_01

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

print("Working directory:", os.getcwd())

## 1. Question

*The research question and the decision it supports*
### Research question:

Can page-level content signals be used to identify content that deserves human review for a possible refresh or optimization?

The analysis focuses on measurable signals such as content age, freshness, search demand, visibility, traffic, engagement, and recent performance.

The goal is not to predict guaranteed traffic growth. The goal is to create a useful ranking that helps a content team decide which pages should be reviewed first.

### Decision supported:

The model supports one practical decision:

> Which pages should the content team look at first?

The model does not decide what should happen to a page. A person must review the page before any action is taken.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*
### Data scope:

The analysis uses the available page-level content data in the project repository.

The dataset contains page-level search, traffic, engagement, content, freshness, and competition-related fields.

The analysis uses the available observation windows in the dataset.

Client-identifying information is not included in the public interpretation of the results.

Rows are excluded only where required fields are unavailable for a particular analysis. Missing numeric values used by the model are handled through median imputation inside the modeling pipeline.

The dataset is observational. It shows relationships between page characteristics and outcomes; it does not provide a controlled experiment showing that changing one feature causes another outcome.

In [ ]:
# Inspect the repository structure
for path in [
    "data",
    "data/raw",
    "data/processed",
    "work",
    "work/outputs"
]:
    if os.path.exists(path):
        print(f"\n{path}/")
        for item in os.listdir(path)[:30]:
            print("  ", item)

In [ ]:
# Find available CSV files
csv_files = glob.glob("data/raw/*.csv")

print("CSV files found:")
for file in csv_files:
    print(" -", file)

if not csv_files:
    raise FileNotFoundError("No CSV files found in data/raw.")

In [ ]:
# Load the available raw dataset
data_path = csv_files[0]

df = pd.read_csv(data_path)

print("Loaded:", data_path)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

display(df.head())

In [ ]:
print("Dataset columns:")
for column in df.columns:
    print(column)

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Target

The target used for the action-ranking model is `refresh_priority`.

A page is labelled as refresh priority when both of the following conditions are met:

- CTR is below 5%
- engagement rate is below 40%

The target is therefore a constructed decision label rather than a directly observed business outcome.

Because CTR and engagement rate are used to create the label, they are excluded from the model features. This prevents direct target leakage.

### Features

The model uses page-level signals covering:

- search demand
- competition
- CPC
- content depth
- impressions
- clicks
- sessions
- engagement
- AI sessions
- visibility duration
- recent traffic
- content age
- days since the last update
- recent trend

### Model

The main model is logistic regression.

The preprocessing pipeline uses median imputation for missing numeric values and standardization before fitting the classifier.

### Baseline

A dummy classifier is used as a simple baseline. It predicts the majority class from the training data.

The model and baseline are evaluated on exactly the same held-out test set.

### Validation

Pages are split using client groups rather than randomly mixing pages from the same client between training and testing.

This reduces the risk that the model is evaluated on pages from clients it has effectively already seen during training.

### Leakage checks

CTR and engagement rate are excluded because they directly define the target.

The client identifier is used only for grouped splitting and is not included as a model feature.

The final queue model is trained on all available rows only after the held-out validation comparison has been completed.

In [ ]:
# Recreate the target used in the action-playbook work.

required_target_columns = [
    "ctr",
    "engagement_rate"
]

missing_target_columns = [
    col for col in required_target_columns
    if col not in df.columns
]

if missing_target_columns:
    raise KeyError(
        f"Missing target columns: {missing_target_columns}"
    )

df["refresh_priority"] = (
    (df["ctr"] < 0.05) &
    (df["engagement_rate"] < 0.40)
).astype(int)

print("Target distribution:")
display(df["refresh_priority"].value_counts())

In [ ]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

missing_features = [
    col for col in features
    if col not in df.columns
]

if missing_features:
    raise KeyError(
        f"Missing expected features: {missing_features}"
    )

print("Number of model features:", len(features))

In [ ]:
# Confirm that leakage variables are not part of the model.

leakage_columns = {
    "ctr",
    "engagement_rate"
}

feature_leakage = leakage_columns.intersection(features)

print("Leakage columns found in features:", feature_leakage)

if feature_leakage:
    raise ValueError(
        f"Potential target leakage detected: {feature_leakage}"
    )

print("Leakage check passed.")

### Grouped train/test split

In [ ]:
if "client_id" not in df.columns:
    raise KeyError("client_id is required for grouped validation.")

X = df[features].copy()
y = df["refresh_priority"].copy()
groups = df["client_id"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", f"{len(X_train):,}")
print("Test rows:", f"{len(X_test):,}")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Shared clients:", len(train_clients & test_clients))

### Baseline

In [ ]:
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_metrics = {
    "Accuracy": accuracy_score(y_test, baseline_predictions),
    "Precision": precision_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        baseline_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        baseline_predictions,
        zero_division=0
    )
}

baseline_metrics

### Model

In [ ]:
model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

model_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        model_predictions
    ),
    "Precision": precision_score(
        y_test,
        model_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        model_predictions,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        model_predictions,
        zero_division=0
    )
}

model_metrics

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*
## Results:

The model is compared with the baseline using the same held-out test set.

F1 is the main comparison metric because the action-ranking task needs to balance identifying priority pages with avoiding unnecessary recommendations.

Accuracy, precision, and recall are also reported so the result is not reduced to one number.

The results should be interpreted as validation performance for this dataset and target definition. They are not evidence that the model will perform identically on a different portfolio.

In [ ]:
results = pd.DataFrame(
    [
        baseline_metrics,
        model_metrics
    ],
    index=["Baseline", "Logistic Regression"]
)

display(results.round(4))

In [ ]:
# Show the confusion matrix for the main model

cm = confusion_matrix(
    y_test,
    model_predictions
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(cm_df)

### Interpretation

The useful comparison is whether the logistic-regression model improves on the simple majority-class baseline on the same test set.

If the model provides a better F1 score than the baseline, that supports using the model as a ranking signal for this specific decision problem.

It still does not prove that a page with a high score will improve after a refresh. The model predicts similarity to the constructed priority label, not future SEO performance.

### Model scoring for the action queue

In [ ]:
# Train the final queue model on all available rows.
# This is done only after the validation comparison above.

queue_model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

queue_model.fit(X, y)

df["priority_score"] = (
    queue_model.predict_proba(X)[:, 1]
)

print("Priority score summary:")
display(df["priority_score"].describe())

### Reason codes

In [ ]:
stale_threshold = (
    df["days_since_last_update"].quantile(0.75)
)

demand_threshold = (
    df["search_volume"].quantile(0.75)
)

impression_threshold = (
    df["impressions_90d"].quantile(0.50)
)

def build_reason_codes(row):
    reasons = []

    if row["days_since_last_update"] >= stale_threshold:
        reasons.append("STALE")

    if row["trend_pct"] < 0:
        reasons.append("DECLINING")

    if row["search_volume"] >= demand_threshold:
        reasons.append("HIGH_DEMAND")

    if row["impressions_90d"] >= impression_threshold:
        reasons.append("VISIBLE")

    if len(reasons) >= 2:
        reasons.append("MULTIPLE_SIGNALS")

    return ", ".join(reasons) if reasons else "REVIEW"

df["reason_codes"] = df.apply(
    build_reason_codes,
    axis=1
)

In [ ]:
def assign_action(row):
    reasons = row["reason_codes"]

    if "STALE" in reasons and "DECLINING" in reasons:
        return "Refresh and review"

    if "DECLINING" in reasons and "HIGH_DEMAND" in reasons:
        return "Investigate and refresh"

    if "STALE" in reasons:
        return "Freshness review"

    if "HIGH_DEMAND" in reasons and "VISIBLE" in reasons:
        return "Optimization review"

    if "VISIBLE" in reasons:
        return "Content review"

    return "Monitor"

df["recommended_action"] = df.apply(
    assign_action,
    axis=1
)

In [ ]:
queue_columns = [
    "content_id",
    "client_id",
    "priority_score",
    "recommended_action",
    "reason_codes",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "trend_pct",
    "days_since_last_update",
    "word_count",
    "content_age_days"
]

action_queue = (
    df[queue_columns]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .reset_index(drop=True)
)

action_queue.insert(
    0,
    "priority_rank",
    range(1, len(action_queue) + 1)
)

display(action_queue.head(20))

## 5. Limitations

*What this work cannot claim.*

This work has several important limits.

### Constructed target

The target is based on CTR and engagement thresholds. It is not a direct measurement of whether a page actually needs a content refresh.

### Observational data

The analysis observes relationships in existing content data. It does not establish that changing word count, freshness, position, or another feature will cause better performance.

### Dataset-specific results

The model was trained and tested on this dataset. Performance may change on another content portfolio, another time period, or a different target definition.

### Grouped validation

Client groups were separated during validation to reduce leakage across clients. This provides a more conservative test than randomly mixing pages, but it does not remove every possible source of bias.

### Score interpretation

The priority score is a model probability associated with the constructed label. It should not be interpreted as a probability that a refresh will increase traffic.

### Human judgment

The model cannot evaluate factual accuracy, search intent changes, brand requirements, business strategy, legal concerns, or the actual quality of a page.

### Public interpretation

The findings should therefore be described as observed, measured, directional, and decision-support signals rather than guaranteed SEO outcomes.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The recommendations below come from the measured patterns in the analysis and the action playbook developed in Week 7.

### 1. Refresh mature pages before they decay

Older pages with previous visibility are strong candidates for human review, especially when they are stale or declining.

The data showed a clear difference between mature stale content and older content that had recently been refreshed.

### 2. Improve page-one click capture

Pages that already have visibility can be useful optimization targets.

Improving titles, descriptions, relevance, and clarity may be more practical than rebuilding pages with little existing visibility.

### 3. Prioritize lower-competition commercial and transactional topics

The portfolio showed stronger results for some lower-competition commercial and transactional combinations.

These should be considered when planning new or refreshed content.

### 4. Expand content when the page is genuinely thin

Longer content should not be treated as a target by itself.

Expansion is most useful when a visible page is missing useful sections, examples, comparisons, definitions, or evidence.

### 5. Monitor AI referrals separately

AI referral traffic was small compared with total tracked sessions but showed a different page profile.

It should therefore be monitored as a separate visibility signal rather than treated as a replacement for organic search.

### 6. Use optimization flags as workflow signals

A flag does not mean a page is bad.

A visible page with a measurable issue can be a better optimization candidate because there is enough evidence to understand the problem.

### 7. Do not automate content decisions

The model should rank pages for review.

It should not automatically rewrite, publish, delete, redirect, merge, or approve content.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
output_dir = "work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

print("Output directory:", output_dir)

In [ ]:
# Export model-vs-baseline results

results_path = os.path.join(
    output_dir,
    "capstone_model_vs_baseline.csv"
)

results.to_csv(
    results_path
)

print("Saved:", results_path)

In [ ]:
# Export the action queue

queue_path = os.path.join(
    output_dir,
    "capstone_content_action_queue.csv"
)

action_queue.to_csv(
    queue_path,
    index=False
)

print("Saved:", queue_path)

In [ ]:
# Export the top 50 pages for easier use in the paper

top_50_path = os.path.join(
    output_dir,
    "capstone_top_50_actions.csv"
)

action_queue.head(50).to_csv(
    top_50_path,
    index=False
)

print("Saved:", top_50_path)

In [ ]:
# Export the confusion matrix

confusion_path = os.path.join(
    output_dir,
    "capstone_confusion_matrix.csv"
)

cm_df.to_csv(
    confusion_path
)

print("Saved:", confusion_path)

### Result figure

In [ ]:
plt.figure(figsize=(8, 5))

results["F1"].plot(
    kind="bar"
)

plt.ylabel("F1 score")
plt.xlabel("Model")
plt.title("Model vs Baseline F1 Score")

plt.xticks(rotation=0)
plt.tight_layout()

results_figure_path = os.path.join(
    output_dir,
    "capstone_model_vs_baseline_f1.png"
)

plt.savefig(
    results_figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", results_figure_path)

### Priority-score distribution

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["priority_score"],
    bins=20
)

plt.xlabel("Priority score")
plt.ylabel("Number of pages")
plt.title("Distribution of Content Priority Scores")

plt.tight_layout()

score_figure_path = os.path.join(
    output_dir,
    "capstone_priority_score_distribution.png"
)

plt.savefig(
    score_figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Saved:", score_figure_path)

### Artifact check

In [ ]:
print("Files generated for the paper:\n")

for file in sorted(
    glob.glob("work/outputs/*")
):
    print(
        f"{os.path.basename(file):45s}"
        f" {os.path.getsize(file):,} bytes"
    )

## Final Takeaway

This project turns page-level content data into a practical decision-support workflow.

The logistic-regression model provides a ranked score for pages that resemble the constructed refresh-priority class. Reason codes then provide simple signals that a reviewer can understand.

The validation compares the model with a simple baseline on the same held-out client groups.

The results should be interpreted carefully. A strong model score does not mean that a page will definitely improve after a refresh. It only means that the page matches patterns associated with the target label in this dataset.

The most useful role for the model is therefore to reduce a large content portfolio to a smaller list of pages worth investigating.

A human remains responsible for deciding whether a page should actually be refreshed, expanded, consolidated, redirected, or left unchanged.

# ML-12 — Closing Communication Assets

## 5-minute demo outline

### 1. Problem — 45 seconds
Explain that large content portfolios make it difficult to know which pages deserve attention first.

### 2. Data — 45 seconds
Explain that the project uses page-level search, traffic, engagement, freshness, content, and competition signals.

### 3. Model — 1 minute
Explain the refresh-priority target, leakage checks, grouped validation, logistic regression, and baseline.

### 4. Results — 1 minute
Show the model-vs-baseline result and explain what the score means.

### 5. Action playbook — 1 minute
Show the ranked queue and reason codes.

### 6. Limitations — 30 seconds
Explain that the model supports human review and does not guarantee SEO improvement.

## Social-post cut

I built a machine-learning workflow that turns page-level SEO data into a ranked content-action queue.

Instead of treating a model score as the final answer, the workflow combines validation, simple reason codes, human review rules, and practical recommendations.

The result is a decision-support system that helps identify which content deserves attention first without pretending that correlation equals causation.

## Employer-facing summary

I built and validated a machine-learning workflow that ranks content pages for human review using page-level search, traffic, engagement, freshness, and content signals.

I used grouped validation and a baseline comparison to make the evaluation more realistic and checked for target leakage before interpreting the results.

I then turned the model output into a practical action queue with reason codes, review rules, limitations, and monitoring triggers rather than treating the model as an automated decision-maker.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [✔️] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [✔️] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
